# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided in Croissant schema format:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print basic metadata (title, description, version)
print(f"Dataset name: {getattr(metadata, 'name', None)}\nVersion: {getattr(metadata, 'version', None)}")
print(f"\nDescription: {getattr(metadata, 'description', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Let's display the available record sets and their fields (all referenced by @id)

all_record_sets = list(dataset.record_sets)
print(f"Available Record Sets (by @id):")
for rs in all_record_sets:
    print(f"- {rs['@id']}: {rs.get('name', rs['@id'])}")
    print("  Fields (with their @id):")
    for f in rs['fields']:
        print(f"    - {f['@id']}: {f.get('name', f['@id'])}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Example: Extracting all main record sets into pandas DataFrames

# List all record set @id's for extraction
record_sets_ids = [rs['@id'] for rs in all_record_sets]
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
# Display the first few columns and rows for one relevant record set (choose the first one for demonstration)
if record_sets_ids:
    main_rs_id = record_sets_ids[0]
    print(f"Columns for record set '{main_rs_id}':\n", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data for summary statistics.

We'll use a numeric field and a grouping field based on inspection above. All references are by `@id`.

In [ ]:
# For demonstration, select a numeric field (e.g., mass, abundance, or similar)
# First, display the columns again (replace with your field of interest):

if record_sets_ids:
    # Pick one record set to analyze (main_rs_id from earlier)
    df = dataframes[main_rs_id]
    print("Columns available in the main record set:")
    print(df.columns.tolist())
    
    # Guess typical fields that are numeric, such as 'cr:molecular_weight', 'cr:abundance', etc.
    # For illustration, we'll try 'cr:molecular_weight' (replace by @id as needed)
    candidate_numeric_fields = [col for col in df.columns if 'weight' in col or 'abundance' in col or 'count' in col]
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}\n")
        # Filter for entries where the numeric field > threshold
        # Convert to numeric, ignore errors
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:\n")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records (z-scored):\n")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a field containing 'description', 'id', or 'accession' if exists
        candidate_group_fields = [col for col in df.columns if 'description' in col or 'accession' in col or 'id' in col]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            print(f"\nGrouping by {group_field} (showing mean of numeric field):")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found for EDA in this record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Visualizing the numeric field distribution
if record_sets_ids and candidate_numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # Optional: Boxplot grouped by a categorical/group field
    if candidate_group_fields:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.title(f'{numeric_field} by {group_field} (Filtered)')
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
We demonstrated how to access and explore a multi-record-set FAIR² dataset using `mlcroissant`, including how to reference record sets and fields by their `@id`s. We conducted basic exploratory analysis and showed how to filter, normalize, and visualize the data for downstream analysis.